In [13]:
import sys
sys.path.append('..')
from transformers import AutoModelForMaskedLM, AutoTokenizer
from prosst.structure.get_sst_seq import SSTPredictor
from Bio import SeqIO
import torch
import pandas as pd
from scipy.stats import spearmanr

In [14]:
import os
os.environ["http_proxy"] = "http://127.0.0.1:15777"
os.environ["https_proxy"] = "http://127.0.0.1:15777"

Load ProSST from Hugging Face. 
(You may need to configure the proxy settings if you are in a region that cannot access the hugging face model.)

In [16]:
prosst_model = AutoModelForMaskedLM.from_pretrained("AI4Protein/ProSST-2048", trust_remote_code=True)
prosst_tokenizer = AutoTokenizer.from_pretrained("AI4Protein/ProSST-2048", trust_remote_code=True)

# Apply the same weight-tying fix used in the benchmark script
prosst_model.cls.predictions.decoder.weight = prosst_model.prosst.embeddings.word_embeddings.weight
assert prosst_model.cls.predictions.decoder.weight.data_ptr() == prosst_model.prosst.embeddings.word_embeddings.weight.data_ptr()
print("Decoder/embedding weights manually tied ✓")

Loading weights: 100%|██████████| 275/275 [00:00<00:00, 15160.80it/s]
ProSSTForMaskedLM LOAD REPORT from: AI4Protein/ProSST-2048
Key                            | Status  | 
-------------------------------+---------+-
cls.predictions.decoder.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Decoder/embedding weights manually tied ✓


Load strcuture quantizer

In [17]:
predictor = SSTPredictor(structure_vocab_size=2048)

---------- Load Model on cuda ----------
MODEL: 5.90M parameters


Read protein sequence

In [25]:
residue_sequence = str(SeqIO.read('example_data/residue_sequence/RPC1_LAMBD_Li_2019_high-expression.fasta', 'fasta').seq)
    

Quantize the structure

In [26]:
import importlib
import prosst.structure.get_sst_seq as sst
importlib.reload(sst)
from prosst.structure.get_sst_seq import SSTPredictor

predictor = SSTPredictor(structure_vocab_size=2048, num_processes=1, num_threads=1)
structure_sequence = predictor.predict_from_pdb(
    '../ProteinGym_v1_AlphaFold2_PDB/proteingym_pdb/RPC1_LAMBD_Li_2019_high-expression.pdb'
 )[0]['2048_sst_seq']

---------- Load Model on cuda ----------
MODEL: 5.90M parameters
---------- Building Subgraphs ----------


  0%|          | 0/1 [00:00<?, ?it/s]

[1/1] Built subgraphs for RPC1_LAMBD_Li_2019_high-expression.pdb (237 residues)


100%|██████████| 1/1 [00:00<00:00, 10.28it/s]


In [21]:
print(structure_sequence)

[1809, 1809, 1809, 740, 77, 204, 1773, 1695, 1627, 449, 449, 1879, 1245, 1961, 960, 1951, 1400, 874, 408, 802, 385, 1064, 913, 1036, 1674, 174, 1634, 1731, 836, 318, 1736, 174, 395, 2003, 1625, 1887, 1815, 1656, 1625, 1829, 882, 546, 2043, 2003, 2003, 2003, 1656, 4, 412, 395, 1815, 1193, 24, 857, 1366, 1366, 1366, 1123, 1149, 681, 2036, 141, 1527, 1193, 604, 2001, 87, 1506, 1077, 24, 703, 2022, 836, 1008, 1284, 837, 1064, 1400, 608, 125, 720, 191, 1638, 1032, 580, 1104, 1104, 682, 953, 415, 953, 1456, 1900, 270, 270, 270, 1836, 993, 458, 664, 303, 1211, 1069, 1069, 760, 1690, 1431, 151, 224, 91, 732, 782, 331, 964, 2012, 37, 2024, 2024, 993, 993, 993, 1139, 993, 18, 1510, 18, 1675, 186, 822, 1069, 1690, 1735, 1700, 303, 1753, 1121, 492, 120, 905, 1160, 781, 726, 1963, 1482, 1069, 414, 1484, 1416, 527, 1092, 1892, 974, 100, 907, 737, 737, 1881, 1881, 1484, 1385, 1255, 1092, 919, 815, 508, 36, 1834, 276, 1834, 508, 1722, 1892, 446, 975, 792, 1903, 1183, 1945, 815, 752, 1853, 1853, 135, 3

Shift the quantized structure sequence, (for 3 special tokens [CLS], [SEP] and [PAD])

In [27]:
structure_sequence_offset = [i + 3 for i in structure_sequence]

Prepare model input

In [28]:
tokenized_res = prosst_tokenizer([residue_sequence], return_tensors='pt')
input_ids = tokenized_res['input_ids']
attention_mask = tokenized_res['attention_mask']
structure_input_ids = torch.tensor([1, *structure_sequence_offset, 2], dtype=torch.long).unsqueeze(0)

Inferece 

In [29]:
with torch.no_grad():
    outputs = prosst_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        ss_input_ids=structure_input_ids
    )
logits = torch.log_softmax(outputs.logits[:, 1:-1], dim=-1).squeeze()

Score mutants

In [30]:
df = pd.read_csv("example_data/substitutions/RPC1_LAMBD_Li_2019_high-expression.csv")
mutants = df['mutant'].tolist()

In [31]:
vocab = prosst_tokenizer.get_vocab()
pred_scores = []
for mutant in mutants:
    mutant_score = 0
    for sub_mutant in mutant.split(":"):
        wt, idx, mt = sub_mutant[0], int(sub_mutant[1:-1]) - 1, sub_mutant[-1]
        pred = logits[idx, vocab[mt]] - logits[idx, vocab[wt]]
        mutant_score += pred.item()
    pred_scores.append(mutant_score)

Compute the spearman correlation

In [32]:
spearmanr(pred_scores, df['DMS_score'])

SignificanceResult(statistic=0.5218617493617493, pvalue=6.545932214765639e-26)

In [3]:
from pathlib import Path
import shutil
import sys

if ".." not in sys.path:
    sys.path.append("..")

from zero_shot.precompute_soft_embeddings import precompute_soft_embeddings

pdb_src_candidates = [
    Path("example_data/GRB2_HUMAN_Faure_2021.pdb"),
    Path("zero_shot/example_data/GRB2_HUMAN_Faure_2021.pdb"),
]
pdb_src = next((p for p in pdb_src_candidates if p.exists()), None)
if pdb_src is None:
    raise FileNotFoundError("Could not find GRB2_HUMAN_Faure_2021.pdb in expected paths")

single_pdb_dir = Path("example_data/_single_pdb_test")
single_pdb_dir.mkdir(parents=True, exist_ok=True)
single_pdb_path = single_pdb_dir / pdb_src.name
shutil.copy2(pdb_src, single_pdb_path)

output_dir = Path("soft_embeddings_test/T_0_single")
cache_dir = Path("cache_subgraphs_test")
if output_dir.exists():
    shutil.rmtree(output_dir)

precompute_soft_embeddings(
    pdb_dir=str(single_pdb_dir),
    output_dir=str(output_dir),
    temperature=0,
    structure_vocab_size=2048,
    device=("cuda" if __import__("torch").cuda.is_available() else "cpu"),
    num_processes=1,
    num_threads=1,
    cache_subgraph_dir=str(cache_dir),
)

created = sorted(p.name for p in output_dir.glob("*.pt"))
print("Created files:", created)
print("Output dir:", output_dir.resolve())

---------- Load Model on cuda ----------
MODEL: 5.90M parameters
Building subgraphs for 2 PDBs...
---------- Building Subgraphs ----------


  0%|          | 0/2 [01:02<?, ?it/s]


---------- Load Model on cuda ----------
MODEL: 5.90M parameters
Building subgraphs for 2 PDBs...
---------- Building Subgraphs ----------


  0%|          | 0/2 [01:02<?, ?it/s]


KeyboardInterrupt: 

In [12]:
from pathlib import Path
import importlib
import os
import shutil
import sys
import torch

if ".." not in sys.path:
    sys.path.append("..")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import prosst.structure.get_sst_seq as sst
import zero_shot.precompute_soft_embeddings as pse
importlib.reload(sst)
importlib.reload(pse)
from zero_shot.precompute_soft_embeddings import precompute_soft_embeddings

pdb_dir_candidates = [
    Path("../ProteinGym_v1_AlphaFold2_PDB/proteingym_pdb"),
]
pdb_dir = next((p for p in pdb_dir_candidates if p.exists()), None)
if pdb_dir is None:
    raise FileNotFoundError("Could not find ProteinGym PDB directory")

output_dir_all = Path("soft_embeddings_test/T_0")
cache_dir_all = Path("cache_subgraphs_all")

if output_dir_all.exists():
    shutil.rmtree(output_dir_all)

# Free as much GPU memory as possible before precompute
if torch.cuda.is_available():
    for name in ["prosst_model", "outputs", "logits", "input_ids", "attention_mask", "structure_input_ids"]:
        if name in globals():
            del globals()[name]
    torch.cuda.empty_cache()

total_pdb = len(list(pdb_dir.glob("*.pdb")))
print(f"Total PDB files: {total_pdb}")
print("Starting precompute... try CUDA first with low-memory settings.")

try:
    precompute_soft_embeddings(
        pdb_dir=str(pdb_dir),
        output_dir=str(output_dir_all),
        temperature=0,
        structure_vocab_size=2048,
        device=("cuda" if torch.cuda.is_available() else "cpu"),
        num_processes=1,
        num_threads=1,
        max_batch_nodes=1200,
        chunk_size=512,
        cache_subgraph_dir=str(cache_dir_all),
    )
except torch.cuda.OutOfMemoryError:
    print("CUDA OOM hit. Clearing cache and retrying on CPU...")
    if output_dir_all.exists():
        shutil.rmtree(output_dir_all)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    precompute_soft_embeddings(
        pdb_dir=str(pdb_dir),
        output_dir=str(output_dir_all),
        temperature=0,
        structure_vocab_size=2048,
        device="cpu",
        num_processes=1,
        num_threads=1,
        max_batch_nodes=1200,
        chunk_size=512,
        cache_subgraph_dir=str(cache_dir_all),
    )

created_all = sorted(p.name for p in output_dir_all.glob("*.pt"))
print("Generated embeddings:", len(created_all))
print("First 5 files:", created_all[:5])
print("Output dir:", output_dir_all.resolve())

Total PDB files: 217
Starting precompute... try CUDA first with low-memory settings.
---------- Load Model on cuda ----------
MODEL: 5.90M parameters
Loading cached subgraphs from cache_subgraphs_all...
---------- Load Subgraphs ----------


100%|██████████| 217/217 [00:04<00:00, 50.59it/s]


Running GVP encoder once to collect node embeddings...


  2%|▏         | 1/66 [00:00<00:41,  1.56it/s]


CUDA OOM hit. Clearing cache and retrying on CPU...
---------- Load Model on cpu ----------
MODEL: 5.90M parameters
Loading cached subgraphs from cache_subgraphs_all...
---------- Load Subgraphs ----------


100%|██████████| 217/217 [00:04<00:00, 52.96it/s]


Running GVP encoder once to collect node embeddings...


 35%|███▍      | 23/66 [03:25<06:24,  8.94s/it]


KeyboardInterrupt: 